# Transaction Foundation Model on Ray — Part 3: Tokenize with NVIDIA's tokenizer

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min (full: longer — encodes the whole train split)

---

In Part 2 we regenerated NVIDIA's temporal split. Now we **tokenize with NVIDIA's own `FinancialTabularTokenizer`** (vendored into `src/nvidia_tokenizer/`) and build the **pretraining corpus**: each card's transactions become a token sequence, packed into 4096-token windows for the causal decoder to learn from in Part 4. Using NVIDIA's tokenizer (not a reimplementation) is what makes our foundation model match/beat theirs.

In [ ]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                                        # same knob as Part 2; full = every card
cfg = load_scale(SCALE)
paths = artifact_paths(get_demo_base_dir(), SCALE)

## NVIDIA's transaction tokenizer

We use NVIDIA's `FinancialTabularTokenizer` (Apache-2.0, vendored verbatim into `src/nvidia_tokenizer/`). Each transaction becomes **12 field tokens** drawn from a **shared vocabulary of 6,251 tokens**:

- **Merchant** → hashed into 2,000 buckets (merchant names are open-ended; hashing keeps the vocab bounded).
- **MCC, twice** → the raw code (`MCC_5411`) plus a coarse industry category derived from it (`CAT_RETAIL`) — the category hierarchy.
- **Time** → hour, day-of-week, and month tokens on every transaction (the burstiness from Part 2 is why time gets explicit fields; a gap-since-last-transaction tokenizer also ships but the blueprint leaves it off, and so do we).
- **Amount** → bucketed at fixed dollar thresholds (\$0/10/50/100/500/1K/5K, from Part 2's orders-of-magnitude spread).
- Plus card index, chip/swipe/online channel, ZIP3 region, state, and customer id.

A card's history becomes a single stream — `<bos> [txn₁ tokens] <sep> [txn₂ tokens] <sep> … <eos>` — packed into fixed **4096-token** sequences (~315 transactions each, NVIDIA's chunking). That flat stream is exactly what the Llama causal decoder pretrains on by next-token prediction (Part 4).

## Build the pretraining corpus

We build the corpus from the temporal **train** split (Part 2). For each card: restore its transactions to time order, derive the 12 token strings per transaction, pack every `chunk` of them (~315 at full) into one `<bos> … <sep> … <eos>` sequence, and encode to `seq_len` token ids. The result is three arrays the pretrainer reads directly: `ids.npy` (`(n_sequences, seq_len)` int32, right-padded), `attn.npy` (the attention mask), and `vocab.json`.

The work is **independent per card** — the vocabulary is static, and no card's tokens depend on any other card. That makes it a textbook `groupby → map_groups` job: Ray Data shuffles the 19.5M train rows into ~6.1K card groups and runs one card at a time across the CPU workers, using the pandas mirror of NVIDIA's tokenizer (`src/nvtokenize_cpu.py`) that Stage 0 of the port proved **byte-identical** to their cuDF/GPU implementation — hashes, token strings, and vocabulary all verified equal, so the corpus that comes out is bit-for-bit the corpus NVIDIA's single-GPU code produces.

In [ ]:
from src.nvsplit import train_parquet_files
from src.nvcorpus import tokenize_card_group, fresh_seqs_dir, assemble_corpus

tk = cfg["tokenize"]
seq_len = tk["seq_len"]                        # tokens per pretrain sequence (full 4096 = NVIDIA)
chunk = max(1, seq_len // 13)                  # ~13 tokens/txn → transactions packed per sequence
max_seq = tk.get("max_pretrain_windows")       # cap #sequences (mini/CI); None = every sequence

# One card at a time, across the cluster: shuffle the train split into per-card groups,
# tokenize + chunk + encode each group on a CPU worker, then assemble the arrays in a
# deterministic (user, card, chunk) order. Re-runs reuse the cached corpus.
if not os.path.exists(os.path.join(paths["nvcorpus"], "ids.npy")):
    seqs_dir = fresh_seqs_dir(paths["nvcorpus"])
    ray.data.read_parquet(train_parquet_files(paths["nvsplit"])) \
        .groupby(["User", "Card"]) \
        .map_groups(lambda g: tokenize_card_group(g, seq_len, chunk),
                    batch_format="pandas") \
        .write_parquet(seqs_dir)
    meta = assemble_corpus(seqs_dir, paths["nvcorpus"], seq_len, max_seq)
    print(json.dumps(meta, indent=2))
else:
    print("corpus already built at", paths["nvcorpus"])

## The tokenized corpus

Let's look at what we built: the number of sequences and the fraction of real (non-pad) tokens, then decode the first sequence back to readable tokens using the same tokenizer. Each transaction's field tokens appear in order, framed by `<bos>` / `<sep>` / `<eos>` (merchant/MCC are hashed, so they read as bucket ids).

In [3]:
from src.nvidia_tokenizer import FinancialTabularTokenizer

ids = np.load(os.path.join(paths["nvcorpus"], "ids.npy"), mmap_mode="r")
attn = np.load(os.path.join(paths["nvcorpus"], "attn.npy"), mmap_mode="r")
with open(os.path.join(paths["nvcorpus"], "vocab.json")) as f:
    vinfo = json.load(f)

print(f"{ids.shape[0]:,} pretrain sequences × {ids.shape[1]} tokens  "
      f"(vocab {vinfo['vocab_size']:,}, pad id {vinfo['pad']})")
print(f"real-token fraction: {float(np.asarray(attn).mean()):.3f}\n")

# Decode sequence 0 back to readable tokens (drop right-padding).
tok = FinancialTabularTokenizer(merchant_hash_size=2000, category_hierarchy=True,
                                temporal_encoding=True)
row = np.asarray(ids[0])
real = [int(t) for t in row if int(t) != vinfo["pad"]]
decoded = tok.decode(real).split()
print(f"sequence 0 — {len(real)} real tokens; first ~2 transactions:")
print("  " + " ".join(decoded[:32]))

/home/ray/anaconda3/lib/python3.12/site-packages/cudf/utils/gpu_utils.py:162: UserWarning: No NVIDIA GPU detected
  warnings.warn("No NVIDIA GPU detected")


64,335 pretrain sequences × 4096 tokens  (vocab 6,251, pad id 0)
real-token fraction: 0.963

sequence 0 — 4096 real tokens; first ~2 transactions:
  <bos> AMT_3 MERCH_667 CAT_RETAIL MCC_5300 HOUR_06 DOW_6 MONTH_09 CARD_0 CHIP_SWIPE ZIP3_917 STATE_CA CUST_0 <sep> AMT_1 MERCH_869 CAT_RETAIL MCC_5411 HOUR_06 DOW_6 MONTH_09 CARD_0 CHIP_SWIPE ZIP3_917 STATE_CA CUST_0 <sep> AMT_3 MERCH_869 CAT_RETAIL MCC_5411 HOUR_06


The vocabulary is fixed at **6,251 tokens**, so the model knows its size up front — no data scan, no cold-start. Special tokens (`<pad>`/`<bos>`/`<eos>`/`<sep>`/`<unk>`) come first, then each field owns a disjoint block of ids (hashed merchant buckets, category hierarchy, temporal fields, amount bins).

In [4]:
from src.nvidia_tokenizer import FinancialTabularTokenizer

tok = FinancialTabularTokenizer(merchant_hash_size=2000, category_hierarchy=True,
                                temporal_encoding=True)
vocab = tok.vocab
specials = [t for t in ("<pad>", "<bos>", "<eos>", "<sep>", "<unk>") if t in vocab]
print(f"shared vocabulary: {tok.vocab_size:,} tokens  ({len(specials)} special + per-field blocks)")
print(f"specials: {specials}\n")

# Group the vocab tokens by their field prefix to show the per-field block sizes.
from collections import Counter
prefixes = Counter(t.split("_")[0] for t in vocab if t not in specials and "_" in t)
print("per-field block size in the shared vocab:")
for name, size in sorted(prefixes.items(), key=lambda kv: -kv[1]):
    print(f"  {name:<12} {size:>6,}")

shared vocabulary: 6,251 tokens  (5 special + per-field blocks)
specials: ['<pad>', '<bos>', '<eos>', '<sep>', '<unk>']

per-field block size in the shared vocab:
  CUST          3,000
  MERCH         2,000
  ZIP3          1,000
  MCC             110
  STATE            58
  HOUR             24
  CAT              14
  MONTH            12
  CARD             10
  AMT               7
  DOW               7
  CHIP              4


## Takeaways

**The tokenizer**
- We use NVIDIA's own `FinancialTabularTokenizer` (vendored, Apache-2.0) — merchant hashing, MCC + industry category, hour/day-of-week/month time fields, fixed-threshold amount buckets; fixed vocab **6,251**. Using their exact tokenizer (not a reimplementation) is what lets our foundation model reach their embedding quality.
- Each card's history → a `<bos> … <sep> … <eos>` token stream, packed into 4096-token sequences (~315 transactions), the input to next-token pretraining.

**Ray**
- The corpus build is `groupby(User, Card) → map_groups`: tokenization is independent per card, so the 19.5M-row train split shuffles into ~6.1K card groups that encode in parallel on **CPU workers** — no GPU anywhere in this notebook.
- The CPU tokenizer is a pandas mirror of NVIDIA's cuDF code, proven **byte-identical** before anything relied on it (all 91K merchant hashes, all 12 token columns, the full vocab — `scripts/verify_cpu_tokenizer.py`), and the assembled corpus is bit-for-bit equal to the single-GPU reference build (`scripts/verify_distributed_corpus.py`).
- Same code at `mini` (a few hundred sequences, minutes) and `full` (64,335 × 4096 tokens); only the config changes.

---

## Next

**Part 4 — Pretrain with Ray Train**: next-token prediction over these 4096-token sequences with a Llama causal decoder, plain PyTorch wrapped by Ray Train for DDP, sharding, and checkpointing — the same code from 1 worker to N GPUs.